# Kalshi coverage check

For every trading day from Jan 2022 to Sep 2026, this checks whether a usable 4 pm price exists for
P(hike) and P(cut) at the next Fed meeting (bid-ask spread of 0.20 or less).

- **fresh**: a usable price from that same day
- **carried forward**: the latest usable price is from an earlier day (the table shows how many days old)
- **no usable price / no market**: nothing to use that day

Results are saved to `kalshi_coverage.csv`.

In [ ]:
import io, re, time
import numpy as np, pandas as pd, requests, yfinance as yf

BASE = "https://api.elections.kalshi.com/trade-api/v2"
START, END = "2022-01-03", "2026-09-30"
ET = "America/New_York"
MAX_SPREAD = 0.20

def get_json(url, params=None):
    """GET with retries; returns None for 404 (e.g. a market missing from one endpoint)."""
    for attempt in range(6):
        r = requests.get(url, params=params, timeout=60)
        if r.status_code == 429:            # rate limited: wait and retry
            time.sleep(2 ** attempt); continue
        if r.status_code == 404:
            return None
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"Too many retries: {url}")

# ---- 1. Trading days and the Fed rate (upper bound) on each day ----
spx = yf.download("^GSPC", start=START, end=pd.Timestamp(END) + pd.Timedelta(days=1),
                  progress=False, auto_adjust=False)
days = pd.DatetimeIndex(pd.to_datetime(spx.index.date))
fred_csv = requests.get("https://fred.stlouisfed.org/graph/fredgraph.csv?id=DFEDTARU", timeout=60).text
fred = pd.read_csv(io.StringIO(fred_csv), parse_dates=[0], index_col=0)
# Rate in effect after any decision announced that day (FRED records a change from the next day)
rate = pd.Series(fred.iloc[:, 0].sort_index().reindex(days + pd.Timedelta(days=1), method="ffill").values,
                 index=days).round(2)

# ---- 2. Every Fed meeting: decision time and its markets, keyed by strike X ----
def get_markets(event_ticker):
    for path in ["/historical/markets", "/markets"]:      # settled first, then open/recent
        out, cursor = [], None
        while True:
            p = {"event_ticker": event_ticker, "limit": 200}
            if cursor: p["cursor"] = cursor
            j = get_json(BASE + path, p) or {}
            out += j.get("markets", [])
            cursor = j.get("cursor")
            if not cursor: break
        if out: return out
    return []

def strike_of(m):
    """Strike X from the ticker, e.g. FED-22DEC-T4.25 -> 4.25, FED-22MAR-T1 -> 1.0."""
    found = re.search(r"-T(\d+(?:\.\d+)?)$", m["ticker"])
    return round(float(found.group(1)), 2) if found else None

events = get_json(f"{BASE}/events", {"series_ticker": "KXFED", "limit": 200})["events"]
meetings = []
for e in events:
    ms = get_markets(e["event_ticker"])
    closes = [m["close_time"] for m in ms if m.get("close_time")]
    if not closes: continue
    meetings.append({"event": e["event_ticker"],
                     "time": pd.Series(pd.to_datetime(closes, utc=True)).median().tz_convert(ET),
                     "markets": {strike_of(m): m["ticker"] for m in ms if strike_of(m) is not None}})
    time.sleep(0.1)
meetings.sort(key=lambda m: m["time"])
print("Meetings found:", len(meetings))

# ---- 3. Hourly candles (bid/ask) for one market, over the weeks it is the next meeting ----
candles = {}
def close_of(side):
    """Closing price of one side; historical data uses 'close', live data uses 'close_dollars'."""
    for key in ("close_dollars", "close"):
        if side.get(key) is not None:
            return float(side[key])
    raise KeyError("no close price")

def hourly(ticker, start, end):
    if ticker in candles: return candles[ticker]
    p = {"start_ts": int(start.timestamp()), "end_ts": int(end.timestamp()), "period_interval": 60}
    # A market's history can be split between the historical and live endpoints, so ask both and merge
    raw = []
    for url in [f"{BASE}/historical/markets/{ticker}/candlesticks",
                f"{BASE}/series/{ticker.split('-')[0]}/markets/{ticker}/candlesticks"]:
        raw += (get_json(url, p) or {}).get("candlesticks", [])
    rows = []
    for c in raw:
        try:
            bid, ask = close_of(c["yes_bid"]), close_of(c["yes_ask"])
        except (KeyError, TypeError, ValueError):
            continue
        rows.append({"time": pd.to_datetime(c["end_period_ts"], unit="s", utc=True).tz_convert(ET),
                     "bid": bid, "ask": ask})
    df = pd.DataFrame(rows, columns=["time", "bid", "ask"]).drop_duplicates("time").sort_values("time")
    if len(df) and df[["bid", "ask"]].max().max() > 1.5:   # prices in cents -> dollars
        df[["bid", "ask"]] /= 100
    df["mid"], df["spread"] = (df["bid"] + df["ask"]) / 2, df["ask"] - df["bid"]
    candles[ticker] = df
    time.sleep(0.1)
    return df

# ---- 4. For each trading day: is there a usable 4 pm price for P(hike) and P(cut)? ----
mtimes = [m["time"] for m in meetings]
rows = []
for d in days:
    cutoff = d.tz_localize(ET) + pd.Timedelta(hours=16)
    r = float(rate.loc[d])
    row = {"date": d.date(), "year": d.year, "rate": r}
    i = next((k for k, t in enumerate(mtimes) if t > cutoff), None)
    if i is None:
        row["hike_status"] = row["cut_status"] = "no meeting found"
        rows.append(row); continue
    m = meetings[i]
    # start a month before the previous meeting, so a quiet market still has an earlier price to carry forward
    start = (mtimes[i - 1] if i > 0 else cutoff) - pd.Timedelta(days=30)
    row["meeting"] = m["event"]
    for name, strike in [("hike", r), ("cut", round(r - 0.25, 2))]:
        if name == "cut" and r <= 0.25:
            row["cut_status"] = "no cut possible"; continue
        ticker = m["markets"].get(strike)
        if ticker is None:
            row[f"{name}_status"] = "no market"; continue
        c = hourly(ticker, start, m["time"])
        good = c[(c["time"] <= cutoff) & (c["spread"] <= MAX_SPREAD)]
        if good.empty:
            row[f"{name}_status"] = "no usable price"; continue
        last = good.iloc[-1]
        age = (d.date() - last["time"].date()).days
        row[f"{name}_status"] = "fresh" if age == 0 else "carried forward"
        row[f"{name}_age_days"] = age
        row[f"p_{name}"] = last["mid"] if name == "hike" else 1 - last["mid"]
    rows.append(row)

cov = pd.DataFrame(rows)
cov.to_csv("kalshi_coverage.csv", index=False)

# ---- 5. Summary ----
for name in ["hike", "cut"]:
    print(f"\n{name.upper()} market: % of trading days in each status")
    print((pd.crosstab(cov["year"], cov[f"{name}_status"], normalize="index") * 100).round(1))
    print(f"Days since the price was last updated ({name}): median and max per year")
    print(cov.groupby("year")[f"{name}_age_days"].agg(["median", "max"]))

ok = cov[["hike_status", "cut_status"]].isin(["fresh", "carried forward", "no cut possible"]).all(axis=1)
summary = (ok.groupby(cov["year"]).mean() * 100).round(1)
print("\n% of trading days with BOTH probabilities usable:\n", summary)
print("\nPASS" if (summary >= 70).all() else "\nCHECK: at least one year is below 70% usable")
